In [9]:
import math
import numpy as np
import pandas as pd

import japanize_matplotlib
import matplotlib.pyplot as plt
from scipy.special import gammaln
from scipy.stats import chi2

# 演習問題7.1

In [3]:
df = pd.DataFrame([[13, 5, 5, 3, 4, 18], [378, 200, 151, 47, 31, 33]])
df.columns = [
    f"放射線量_{v}" for v in ["0", "1-9", "10-49", "50-99", "100-199", "200+"]
]
df = pd.concat([df, df.sum().to_frame().T])
df.index = ["白血病", "その他のがん", "合計"]
df.index.name = "死因"

In [4]:
df.T

死因,白血病,その他のがん,合計
放射線量_0,13,378,391
放射線量_1-9,5,200,205
放射線量_10-49,5,151,156
放射線量_50-99,3,47,50
放射線量_100-199,4,31,35
放射線量_200+,18,33,51


In [35]:
df_tmp = df.T
y = (df_tmp["白血病"] / df_tmp["合計"]).values
X = np.stack([np.ones(len(y)), list(range(1, 7))])
n = df_tmp["合計"].values

In [41]:
n

array([391, 205, 156,  50,  35,  51])

In [78]:
def pred_pi(beta, X):
    z = beta @ X
    return np.exp(z) / (1 + np.exp(z))


# 初期値
beta = np.array([0.0, 0.0])

for _ in range(10):
    # piの推定
    pi = pred_pi(beta, X)  # shape (N,)

    # スコアの推定 (2,)
    U = np.sum((y - n * pi) * X, axis=1)  # Xの各行で重み付き和

    # 情報行列の推定 (2x2)
    W = n * pi * (1 - pi)  # shape (N,)
    J = X @ (W[:, None] * X.T)  # 2xN @ N×2 = 2x2
    beta = beta + np.linalg.inv(J) @ U

In [79]:
beta

array([-10.81132773,   0.97888537])

In [50]:
y

array([0.03324808, 0.02439024, 0.03205128, 0.06      , 0.11428571,
       0.35294118])

In [51]:
n

array([391, 205, 156,  50,  35,  51])

In [53]:
y - n * pi

array([-195.46675192, -102.47560976,  -77.96794872,  -24.94      ,
        -17.38571429,  -25.14705882])

In [60]:
n * pi * (1 - pi)

array([97.75, 51.25, 39.  , 12.5 ,  8.75, 12.75])

# 演習問題7.3

In [2]:
data_male = {
    ("Medicine", "S"): [18,16,7,12,24,16,22,12,22,28],
    ("Medicine", "T"): [22,23,17,25,50,21,32,14,34,37],
    ("Arts", "S"): [16,13,11,12,8,11,4,4,None,13],
    ("Arts", "T"): [30,22,25,14,12,20,10,12,None,23],
    ("Science", "S"): [9,9,12,12,20,16,25,32,4,25],
    ("Science", "T"): [14,12,19,15,28,21,31,38,5,31],
    ("Engineering", "S"): [10,7,12,8,5,1,16,19,None,25],
    ("Engineering", "T"): [16,11,15,9,7,2,22,25,None,35],
}

index = [1938,1939,1940,1941,1942,1943,1944,1945,1946,1947]

df_male = pd.DataFrame(data_male, index=index)

data_female = {
    ("Arts","S"): [14, 11, 15, 15, 8, 13, 18, 18, 1, 13],
    ("Arts", "T"): [19, 16, 18, 21, 9, 13, 22, 22, 1, 16],
    ("Science", "S"): [1, 4, 6, 3, 4, 8, 5, 16, 1, 10],
    ("Science", "T"): [1, 4, 7, 3, 4, 9, 5, 17, 1, 10],
}
index = [1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947]
df_female = pd.DataFrame(data_female, index = index)

In [10]:
def stable_sigmoid(z):
    # z = X @ beta の想定
    # 正の大きな値で exp(z) がオーバーフローするのを防ぐため、
    # zの正負で計算式を分ける
    return np.where(
        z >= 0,
        1 / (1 + np.exp(-z)),
        np.exp(z) / (1 + np.exp(z))
    )


def update_beta_binom_logit(X, y, n):
    
    # 設定
    n_iters = 20
    tol = 1e-10
        
    # 初期値
    beta = np.zeros(X.shape[1])
    
    for iter_ in range(n_iters):
        
        # 予測確率の計算 (stable_sigmoidは定義済みと仮定)
        pi = stable_sigmoid(X @ beta)
        
        # 勾配ベクトル (Score function) U
        # U = X.T @ (y - n * pi)
        U = X.T @ (y - n * pi)
        
        # 重み行列 W の対角成分
        # w_i = n_i * pi_i * (1 - pi_i)
        w = n * pi * (1 - pi)
        
        # ヘッセ行列 J = X.T @ W @ X
        # (Wは対角行列なので、ブロードキャストを利用して効率的に計算)
        J = (X.T * w) @ X
        
        # 更新量の計算 (J * delta_beta = U を解く)
        try:
            delta_beta = np.linalg.solve(J, U)
        except np.linalg.LinAlgError:
            print("行列が特異なため、計算を中断します。")
            break
            
        beta_new = beta + delta_beta
        
        # 収束判定
        diff = np.sum((beta_new - beta)**2)
        beta = beta_new
        
        print(f"{iter_ + 1}回目：{beta}")
        
        if diff < tol:
            print(f"{iter_ + 1}回目のiterationで終了")
            break
            
    # 標準誤差の計算
    std_error = np.sqrt(np.diag(np.linalg.inv(J)))
    return beta, std_error, pi

def judge_agains_null_hypo(C, dof, alpha):
    
    lower = chi2.ppf(alpha / 2, dof)
    upper = chi2.ppf(1 - alpha / 2, dof)
    print("lower : ", lower)
    print("upper : ", upper)
    print("C : ", C)

## (a)

In [101]:
# 男性の処理
majors_m = ["Medicine", "Arts", "Science", "Engineering"]
df_male[("all", "S")] = df_male[majors_m].xs("S", axis=1, level=1).fillna(0).sum(axis=1)
df_male[("all", "T")] = df_male[majors_m].xs("T", axis=1, level=1).fillna(0).sum(axis=1)

# 女性の処理
majors_f = ["Arts", "Science"]
df_female[("all", "S")] = df_female[majors_f].xs("S", axis=1, level=1).fillna(0).sum(axis=1)
df_female[("all", "T")] = df_female[majors_f].xs("T", axis=1, level=1).fillna(0).sum(axis=1)

# 結合と整形
df_all = (df_female["all"] + df_male["all"]).reset_index().rename(columns={"index": "year"})
df_dummies = pd.get_dummies(df_all, columns = ["year"], dtype = int, drop_first=True)

In [103]:
X = np.column_stack([np.ones(len(df_dummies)),df_dummies.drop(["S", "T"], axis = 1).values])
y = df_dummies["S"].values
n = df_dummies["T"].values
beta, std_error, pi = update_beta_binom_logit(X, y, n)

pi_tilder = df_all["S"].sum() / df_all["T"].sum()
y_hat = n*pi
C = (y * np.log(y_hat / (n * pi_tilder)) + (n - y) * np.log((n - y_hat) / (n - n * pi_tilder))).sum() * 2
alpha = 0.05
dof = X.shape[1] - 1
print(" ")
judge_agains_null_hypo(C, dof, alpha)

1回目：[ 0.66666667  0.06060606 -0.17161716  0.18390805 -0.15757576  0.35658915
  0.28415301  0.48958333  0.06504065  0.33333333]
2回目：[ 0.6930334   0.06889265 -0.18749828  0.2145494  -0.17251538  0.43412329
  0.33948803  0.61927314  0.07399806  0.40330572]
3回目：[ 0.69314718  0.06899287 -0.18759861  0.21511128 -0.17261274  0.43671578
  0.34092602  0.62612246  0.07410797  0.40546382]
4回目：[ 0.69314718  0.06899287 -0.18759861  0.21511138 -0.17261274  0.43671765
  0.34092659  0.62613647  0.07410797  0.40546511]
5回目：[ 0.69314718  0.06899287 -0.18759861  0.21511138 -0.17261274  0.43671765
  0.34092659  0.62613647  0.07410797  0.40546511]
5回目のiterationで終了
 
lower :  2.7003894999803584
upper :  19.02276779864163
C :  15.042542482746049


## (b)

In [40]:
# 1. 集計と整形を一気に行う
df_wide = (df_male.sum()
           .drop("all")
           .unstack() # stackの代わり。Seriesのマルチインデックスを解除してDataFrame化
           .reset_index().rename(columns={"index":"faculty"})
           )

# 2. ダミー変数化
df_dummies = pd.get_dummies(df_wide, columns=["faculty"], dtype=int, drop_first=True)

# データの準備
X = np.column_stack(
    [np.ones(len(df_dummies)) ,df_dummies[["faculty_Engineering", "faculty_Medicine", "faculty_Science"]].values]
)
y = df_dummies["S"].values
n = df_dummies["T"].values

# ロジスティック回帰の係数推定
beta, std_error, pi = update_beta_binom_logit(X, y, n)

# 尤度比カイ二乗統計量
y_hat = n * pi
pi_tilder = y.sum() / n.sum()
C = 2 * (y * np.log(y_hat / (n * pi_tilder)) + (n - y) * np.log( (n - y_hat) / (n - n*pi_tilder) ) ).sum()
alpha = 0.05
dof = X.shape[1] - 1

# 帰無仮説の判定
judge_agains_null_hypo(C, dof, alpha)

1回目：[0.19047619 0.71093226 0.38406926 0.87494437]
2回目：[0.19105522 0.77907164 0.40008804 0.99309728]
3回目：[0.19105524 0.78011186 0.40012702 0.99678457]
4回目：[0.19105524 0.78011211 0.40012702 0.99678819]
4回目のiterationで終了
lower :  0.21579528262389794
upper :  9.348403604496145
C :  23.2589960500619


/var/folders/g4/gx0098gn4072x6j58_7tw_8c0000gn/T/ipykernel_90457/2719953864.py:3: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  .drop("all")


## (c)

In [58]:
#集計と整形を一気に行う
df_wide = (df_female.sum()
           .drop("all")
           .unstack() # stackの代わり。Seriesのマルチインデックスを解除してDataFrame化
           .reset_index().rename(columns={"index":"faculty"})
           )

#ダミー変数化
df_dummies = pd.get_dummies(df_wide, columns=["faculty"], dtype=int, drop_first=True)

In [68]:
X = np.column_stack(
    [
        np.ones(len(df_dummies)), df_dummies["faculty_Science"].values
    ]
)
n = df_dummies["T"].values
y = df_dummies["S"].values


beta, std_error, pi = update_beta_binom_logit(X, y, n)

# 尤度比カイ二乗統計量
y_hat = n * pi
pi_tilder = y.sum() / n.sum()
C = 2 * (y * np.log(y_hat / (n * pi_tilder)) + (n - y) * np.log( (n - y_hat) / (n - n*pi_tilder) ) ).sum()
alpha = 0.05
dof = X.shape[1] - 1

# 帰無仮説の判定
judge_agains_null_hypo(C, dof, alpha)

1回目：[1.21019108 0.59308761]
2回目：[1.3922789  1.17079337]
3回目：[1.40226452 1.49757921]
4回目：[1.4022947  1.55784294]
5回目：[1.4022947  1.55953473]
6回目：[1.4022947  1.55953602]
6回目のiterationで終了
lower :  0.0009820691171752555
upper :  5.02388618731489
C :  8.816488778764487


## (d)

In [109]:
df_sex_diff= df_male[["Arts", "Science"]].fillna(0) - df_female[["Arts", "Science"]]

In [114]:
#集計と整形を一気に行う
df_wide = (df_sex_diff.sum()
           .unstack() # stackの代わり。Seriesのマルチインデックスを解除してDataFrame化
           .reset_index().rename(columns={"index":"faculty"})
           )

#ダミー変数化
df_dummies = pd.get_dummies(df_wide, columns=["faculty"], dtype=int)

In [115]:
df_dummies

,S,T,faculty_Arts,faculty_Science
0,-34.0,11.0,1,0
1,106.0,153.0,0,1


In [117]:
X = np.column_stack(
    [
        np.ones(len(df_dummies)), df_dummies[["faculty_Arts","faculty_Science"]].values
    ]
)
n = df_dummies["T"].values
y = df_dummies["S"].values

beta, std_error, pi = update_beta_binom_logit(X, y, n)

# 尤度比カイ二乗統計量
y_hat = n * pi
pi_tilder = y.sum() / n.sum()
C = 2 * (y * np.log(y_hat / (n * pi_tilder)) + (n - y) * np.log( (n - y_hat) / (n - n*pi_tilder) ) ).sum()
alpha = 0.05
dof = X.shape[1] - 1

# 帰無仮説の判定
judge_agains_null_hypo(C, dof, alpha)

1回目：[ -1.22875817 -13.13487819   2.        ]
2回目：[-3.89797883e+00 -5.34731389e+06  4.71094174e+00]
行列が特異なため、計算を中断します。


/var/folders/g4/gx0098gn4072x6j58_7tw_8c0000gn/T/ipykernel_90457/2538490864.py:14: RuntimeWarning: overflow encountered in exp
  1 / (1 + np.exp(-z)),


LinAlgError: Singular matrix

In [90]:
df_dummies

,S,T,faculty_Science
0,-34.0,11.0,0
1,106.0,153.0,1
